# Faster autoscaling on Amazon SageMaker realtime endpoints (Application Autoscaling)

---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook.

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints|sm-app_autoscaling_realtime_endpoints.ipynb)


---

In this notebook we show how the new faster autoscaling feature helps scale sagemaker inference endpoints by almost 6x faster than earlier.

We deploy Meta's `Llama3-8B-Instruct` model to an Amazon SageMaker realtime endpoint using Text Generation Inference (TGI) Deep Learning Container (DLC) and apply <span style='color:green'><b>Application Autoscaling</b></span> scaling policies to the endpoint.


<div class="alert alert-block alert-warning">
    Please select <b>m5.2xlarge</b> or larger instance types when running this on Amazon SageMaker Notebook Instance.<br/>
    Select <b>conda_pytorch_p310</b> kernel when running this notebook on Amazon SageMaker Notebook Instance. <br/><br/>
    Ensure python version for the kernel is <b>3.10.x</b> (3.11 is not supported). <br/>
</div>

---

## Prerequisites



<div style="border: 1px solid #f00; border-radius: 5px; padding: 10px; background-color: #fee;">
Before using this notebook please ensure you have access to an active access token from HuggingFace and have accepted the license agreement from Meta.

- **Step 1:** Create user access token in HuggingFace (HF). Refer [here](https://huggingface.co/docs/hub/security-tokens) on how to create HF tokens.
- **Step 2:** Login to [HuggingFace](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/tree/main) and navigate to *Meta-Llama-3-8B-Instruct** home page.
- **Step 3:** Accept META LLAMA 3 COMMUNITY LICENSE AGREEMENT by following the instructions [here](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/tree/main)
- **Step 4:** Wait for the approval email from META (Approval may take any where b/w 1-3 hrs)
</div>

Install packages using uv, an extremely fast python package installer\
Read more about uv here <https://astral.sh/blog/uv>

In [1]:
# ensure python version of the selected kernel is not greater than 3.10
!python --version

Python 3.12.13


In [2]:
# [exec-copy] deps preinstalled in .v3test-venv; skip in-notebook installs
pass

In [3]:
# [exec-copy] no kernel restart under papermill
pass

In [4]:
# [exec-copy] %load_ext rich omitted for headless run
pass

In [5]:
import glob
import json
import os
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from getpass import getpass
from pathlib import Path
from statistics import mean
from uuid import uuid4

import boto3
import botocore
from rich import box, print
from rich.console import Console
from rich.progress import Progress, SpinnerColumn, TimeElapsedColumn
from rich.table import Table

# V3: hosting uses sagemaker-core resources instead of the V2 HuggingFaceModel/Predictor.
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.resources import Model, EndpointConfig, Endpoint
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant

from utils.autoscaling import (
    monitor_scaling_events,
    print_scaling_times,
    test_concurrency_level,
)

from utils.llmperf import (
    print_llmperf_results,
    trigger_auto_scaling,
    monitor_process,
)

/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_data_url" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_data_source" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_description" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/

[07/16/26 13:35:09] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=1943125;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=1943126;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_version" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


## Initiate sagemaker session

In [6]:
# [exec-copy] pin region + explicit role (no notebook-instance context locally)
import os
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-west-2')

sess = Session()
role = "arn:aws:iam::784379639078:role/SageMakerRole"
region = "us-west-2"

boto_session = boto3.Session(region_name=region)

sagemaker_client = boto_session.client("sagemaker")
sagemaker_runtime_client = boto_session.client("sagemaker-runtime")
cloudwatch_client = boto3.client("cloudwatch", region_name=region)

hf_model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

tgi_dlc = image_uris.retrieve(
    framework="huggingface-llm",
    region=region,
    version="2.0.0",
    image_scope="inference",
)

print(f"TGI DLC: \n[b i green]{tgi_dlc}")
print(f"Region: [b blue]{region}")
print(f"Role: [b red]{role}")

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=1943131;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=1943132;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=1943137;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=1943138;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=1943143;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=1943144;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

[07/16/26 13:35:10] INFO     Defaulting to only available Python version: py310                   ]8;id=1943151;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=1943152;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py#615\615]8;;\

                    INFO     Defaulting to only supported image scope: gpu.                       ]8;id=1943158;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=1943159;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py#539\539]8;;\

TGI DLC: 
763104351884.dkr.ecr.us-west-2.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gpu-py310-cu121-ubunt
u22.04

Region: us-west-2

Role: arn:aws:iam::784379639078:role/SageMakerRole

## Deploy model

Create and deploy model using Amazon SageMaker HuggingFace TGI DLC

<https://sagemaker.readthedocs.io/en/stable/api/inference/model.html#sagemaker.model.Model.deploy>

<div class="alert alert-block alert-warning">
<b>NOTE:</b> Remember to copy your Hugging Face Access Token from <a href="https://hf.co/">https://hf.co/</a> before running the below cell.<br/><br/>
Refer <a href="https://huggingface.co/docs/hub/security-tokens">here</a> to learn about creating HF tokens.
</div>

In [7]:
# sagemaker config
# [exec-copy] g5.2xlarge hit InsufficientInstanceCapacity in us-east-1; g5.xlarge (A10G 24GB) also fits Llama3-8B fp16 and has more capacity. Source -v3 keeps g5.2xlarge.
instance_type = "ml.g6.xlarge"
suffix = f"{str(uuid4())[:5]}-{datetime.now().strftime('%d%b%Y')}"
model_name = f"Llama3-8B-fas-{suffix}"
endpoint_config_name = model_name
endpoint_name = model_name
health_check_timeout = 900

# [exec-copy] ungated mirror of Meta-Llama-3-8B-Instruct (identical weights/chat-template);
# avoids the gated EULA. Token optional for an ungated repo.
HF_TOKEN = os.getenv("HUGGING_FACE_HUB_TOKEN", "")
config = {
    "HF_MODEL_ID": "NousResearch/Meta-Llama-3-8B-Instruct",
    "SM_NUM_GPUS": "1",
    "MAX_INPUT_LENGTH": "2048",
    "MAX_TOTAL_TOKENS": "4096",
    "MAX_BATCH_TOTAL_TOKENS": "8192",
    "MESSAGES_API_ENABLED": "true",
    "HUGGING_FACE_HUB_TOKEN": HF_TOKEN,
}

print(f"Creating model: [b green]{model_name}...")
Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(image=tgi_dlc, environment=config),
    execution_role_arn=role,
)

EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=model_name,
            initial_instance_count=1,
            instance_type=instance_type,
            initial_variant_weight=1.0,
            container_startup_health_check_timeout_in_seconds=health_check_timeout,
        )
    ],
)

print(f"Deploying model to endpoint: [b magenta]{endpoint_name}...")
endpoint = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_config_name,
)
endpoint.wait_for_status("InService")

Creating model: Llama3-8B-fas-3825e-16Jul2026...

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


                    INFO     Creating model resource.                                            ]8;id=1943166;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1943167;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#20593\20593]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=1943174;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=1943175;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py#361\361]8;;\

...

                    INFO     Creating endpoint_config resource.                                  ]8;id=1943200;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1943201;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#11069\11069]8;;\

Deploying model to endpoint: Llama3-8B-fas-3825e-16Jul2026...

[07/16/26 13:35:11] INFO     Creating endpoint resource.                                         ]8;id=1943207;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1943208;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10228\10228]8;;\

Output()

[07/16/26 13:40:53] INFO     Final Resource Status: InService                                    ]8;id=1943214;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1943215;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10484\10484]8;;\

## Inference

Invoke and test endpoint using messages API. Refer to HF [Messages API](https://huggingface.co/docs/text-generation-inference/messages_api) for more info.

In [8]:
# Prepare prompt in messages API format
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is deep learning?"},
]

# Generation arguments
parameters = {
    "model": hf_model_id,  # model id is required
    "top_p": 0.6,
    "temperature": 0.9,
    "max_tokens": 512,
    "stop": ["<|eot_id|>"],
}


# V3: predictor.predict(...) -> core Endpoint.invoke(...). The TGI messages API contract
# (request/response JSON shape) is unchanged; we just serialize the request and read the body.
def invoke_messages(endpoint, payload):
    response = endpoint.invoke(
        body=json.dumps(payload),
        content_type="application/json",
        accept="application/json",
    )
    body = response.body.read()
    if isinstance(body, bytes):
        body = body.decode("utf-8")
    return json.loads(body)


chat = invoke_messages(endpoint, {"messages": messages, **parameters})

# Unpack and print response
print(chat["choices"][0]["message"]["content"].strip())

Deep learning is a subfield of machine learning that involves the use of artificial neural networks to model and 
analyze complex data. The term "deep" refers to the fact that these networks typically consist of multiple layers 
of interconnected nodes or "neurons," which process and transform the input data in a hierarchical manner.

Deep learning models are inspired by the structure and function of the human brain, where neurons are organized in 
layers to process and interpret visual, auditory, and other sensory information. In a deep learning model, each 
layer learns to detect and represent more abstract features or patterns in the data, allowing the model to 
recognize and classify complex patterns and relationships.

Some key characteristics of deep learning include:

1. Hierarchical representation: Deep learning models learn to represent data in a hierarchical manner, with early 
layers detecting low-level features and later layers combining these features to detect higher-level patterns.
2. Large datasets: Deep learning models typically require large amounts of training data to learn and improve their
performance.
3. Complex computations: Deep learning models perform complex computations, such as matrix multiplications and 
activation functions, to transform and process the input data.
4. Gradient-based optimization: Deep learning models use gradient-based optimization algorithms, such as stochastic
gradient descent, to adjust the model's parameters and minimize the loss function.

Deep learning has many applications in various fields, including:

1. Computer vision: Deep learning models can be used for image and video analysis, object detection, facial 
recognition, and more.
2. Natural language processing: Deep learning models can be used for language translation, sentiment analysis, text
summarization, and more.
3. Speech recognition: Deep learning models can be used for speech recognition, voice assistants, and more.
4. Robotics: Deep learning models can be used for robot control, object manipulation, and more.

Some popular deep learning architectures include:

1. Convolutional Neural Networks (CNNs): Used for image and video analysis.
2. Recurrent Neural Networks (RNNs): Used for sequential data, such as speech and text.
3. Long Short-Term Memory (LSTM) networks: Used for sequential data with long-term dependencies.
4. Generative Adversarial Networks (GANs): Used for generating new data samples that resemble existing data.

Overall, deep learning is a powerful tool for modeling and analyzing complex data, and has many applications in 
various fields.

## Baseline average latency at various concurrency levels (Optional)

By capturing average latency across various concurrency levels, we can get a fair idea on after how many concurrent request does endpoint performance would degrade significantly.

Having this information can help define values for scaling policy accordingly.

<div class="alert alert-block alert-info">
<b>Running below cell is optional</b><br/><br/>
<b>INFO: ℹ️</b> Signal here is, at a given concurrency level you start to see average latency increase significantly.<br/>
At this concurrency level the endpoint gets overloaded and cannot serve requests in a timely fashion.<br/>
We use these values to set as threshold values for autoscaling.
<br/><br/>
<b>NOTE: ⚠️</b> As concurrent requests to the endpoint increase you might observe <b>ThrottlingException</b> errors as we haven't incorporated exponential backoff and retry mechanisms.
</div>

In [9]:
# Define list of prompts
prompts = [
    "what is deep learning?",
    "what are various inference modes in Amazon SageMaker?",
    "Can I host Large language models on Amazon SageMaker?",
    "Does Amazon SageMaker support TensorRT-LLM?",
    "what is step scaling policy in the context of autoscaling ec2 instances on AWS?",
    "Why is the sky blue?",
    "List 5 benefits of incorporating limes into the diet.",
]

# Test different concurrency levels and measure average latency
concurrency_levels = [10, 50, 75, 100]  # Adjust these values as needed

for concurrency_level in concurrency_levels:
    try:
        avg_latency = test_concurrency_level(
            concurrency_level,
            prompts,
            messages,
            parameters,
            endpoint_name,
            sagemaker_runtime_client,
        )
        print(
            f"[b]Concurrency:[/b] {concurrency_level} requests,"
            f" [b]Average latency:[/b] {avg_latency:.2f} seconds"
        )
    except Exception as e:
        print(f"[b]At Concurrency[/b] {concurrency_level} requests," f"[b]Exception:[/b] \n{e}")
        continue

Concurrency: 10 requests, Average latency: 23.45 seconds

Concurrency: 50 requests, Average latency: 31.82 seconds

Concurrency: 75 requests, Average latency: 36.15 seconds

Request failed: An error occurred (ThrottlingException) when calling the InvokeEndpoint operation (reached max 
retries: 4): None

Request failed: An error occurred (ThrottlingException) when calling the InvokeEndpoint operation (reached max 
retries: 4): None

...

                    WARNING  Connection pool is full, discarding connection:                  ]8;id=1951252;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/urllib3/connectionpool.py\connectionpool.py]8;;\:]8;id=1951253;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/urllib3/connectionpool.py#327\327]8;;\
                             runtime.sagemaker.us-west-2.amazonaws.com. Connection pool size:                      
                             10                                                                                    

                    WARNING  Connection pool is full, discarding connection:                  ]8;id=1951258;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/urllib3/connectionpool.py\connectionpool.py]8;;\:]8;id=1951259;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/urllib3/connectionpool.py#327\327]8;;\
                             runtime.sagemaker.us-west-2.amazonaws.com. Connection pool size:                      
                             10                                                                                    

                    WARNING  Connection pool is full, discarding connection:                  ]8;id=1951264;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/urllib3/connectionpool.py\connectionpool.py]8;;\:]8;id=1951265;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/urllib3/connectionpool.py#327\327]8;;\
                             runtime.sagemaker.us-west-2.amazonaws.com. Connection pool size:                      
                             10                                                                                    

                    WARNING  Connection pool is full, discarding connection:                  ]8;id=1951270;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/urllib3/connectionpool.py\connectionpool.py]8;;\:]8;id=1951271;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/urllib3/connectionpool.py#327\327]8;;\
                             runtime.sagemaker.us-west-2.amazonaws.com. Connection pool size:                      
                             10                                                                                    

Concurrency: 100 requests, Average latency: 47.40 seconds

---

## Apply Autoscaling policies to the endpoint

Apply Application Autoscaling Policy to endpoint

1. Register Scalable Target

In [10]:
variant_name = "AllTraffic"
as_min_capacity = 1
as_max_capacity = 2

resource_id = f"endpoint/{endpoint_name}/variant/{variant_name}"

autoscaling_client = boto3.client("application-autoscaling", region_name=region)

# Register scalable target
scalable_target = autoscaling_client.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=as_min_capacity,
    MaxCapacity=as_max_capacity,  # Replace with your desired maximum instances
)

scalable_target_arn = scalable_target["ScalableTargetARN"]
print(f"Resource ID: [b blue]{resource_id}")
print(f"Scalable_target_arn:\n[b green]{scalable_target_arn}")

Resource ID: endpoint/Llama3-8B-fas-3825e-16Jul2026/variant/AllTraffic

Scalable_target_arn:
arn:aws:application-autoscaling:us-west-2:784379639078:scalable-target/056m0f967abb5fbc41bab0d221a4efac572b

## Use the latest high-resolution Metrics to trigger auto-scaling

- New feature introduces a new <span style='color:green'><b>PredefinedMetricType</b></span> for scaling policy configuration i.e. <span style='color:green'><b>SageMakerVariantConcurrentRequestsPerModelHighResolution</b></span> to trigger scaling actions.
- Creating a scaling policy with this metric type will create cloudwatch alarms that track a new metric called <span style='color:green'><b>ConcurrentRequestsPerModel</b></span>.
- These high-resolution metrics are published at sub-minute intervals (10s intervals to CW + any additional jitter + delays)
- We should observe significant improvement in scale out times with this new metric


### Steps to create Application autoscaling policy

- Create scaling policy
  - Set `PolicyType` to `TargetTrackingScaling`
  - Set `TargetValue` to `5.0`. i.e., Scaling triggers when endpoint receives 5 `ConcurrentRequestsPerModel`
  - Set `PredefinedMetricType` to `SageMakerVariantConcurrentRequestsPerModelHighResolution`
  - Set `ScaleInCoolDown` and `ScaleOutCoolDown` values to `300` seconds

In [11]:
# Create Target Tracking Scaling Policy
target_tracking_policy_response = autoscaling_client.put_scaling_policy(
    PolicyName="SageMakerEndpointScalingPolicy",
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 5.0,  # Scaling triggers when endpoint receives 5 ConcurrentRequestsPerModel
        "PredefinedMetricSpecification": {
            "PredefinedMetricType": "SageMakerVariantConcurrentRequestsPerModelHighResolution"
        },
        "ScaleInCooldown": 300,  # Cooldown period after scale-in activity
        "ScaleOutCooldown": 300,  # Cooldown period after scale-out activity
    },
)

# print(target_tracking_policy_response)
print(f"[b]Policy ARN:[/b] [i blue]{target_tracking_policy_response['PolicyARN']}")

# print Cloudwatch Alarms
alarms = target_tracking_policy_response["Alarms"]

for alarm in alarms:
    print(f"[b]Alarm Name:[/b] [b magenta]{alarm['AlarmName']}")
    # print(f"[b]Alarm ARN:[/b] [i green]{alarm['AlarmARN']}[/i green]")
    print("===" * 15)

Policy ARN: 
arn:aws:autoscaling:us-west-2:784379639078:scalingPolicy:0f967abb-5fbc-41ba-b0d2-21a4efac572b:resource/sagemaker/en
dpoint/Llama3-8B-fas-3825e-16Jul2026/variant/AllTraffic:policyName/SageMakerEndpointScalingPolicy

Alarm Name: 
TargetTracking-endpoint/Llama3-8B-fas-3825e-16Jul2026/variant/AllTraffic-AlarmHigh-9a27e1d2-f164-496c-8729-4bc67586
c394

=============================================

Alarm Name: 
TargetTracking-endpoint/Llama3-8B-fas-3825e-16Jul2026/variant/AllTraffic-AlarmLow-24d3971c-c89a-4b91-b395-1a2ec420f
c76

=============================================

## Trigger autoscaling action

### LLMPerf to generate traffic to the endpoint

Refer to <https://github.com/philschmid/llmperf> for more details on LLMPerf.

Run the LLMPerf traffic generation script in the background using `subprocess.Popen`

<div class="alert alert-block alert-info">
<b>INFO:ℹ️</b> Refer to <a href="utils/llmperf.py"><b>utils/llmperf.py</b></a> for <span style='color:red'>trigger_autoscaling</span> function implementation
</div>

### Monitor Scale-Out Alarm Trigger times and scaling event times

As llmperf generates traffic to the endpoint continuously this trigger auto-scaling.

The `monitor_scaling_events` function does the following:
- Calculates time taken for alarm to go into InAlarm state.
- checks if alarm is InAlarm state. If yes, then starts the scaling timer
- continuously monitors the `DesiredInstanceCount` property of the endpoint
  - waits till `CurrentInstanceCount == DesiredInstanceCount` and `EndpointStatus` is `InService`
- Calculates time taken to scale out instances prints the times in a table

The below cell triggers auto scaling action and calls the monitor_scaling_events immediately on the AlarmHigh

<div class="alert alert-block alert-info">
<b>INFO: ℹ️</b> Refer to <a href="utils/autoscaling.py"><b>utils/autoscaling.py</b></a> for <span style='color:red'>monitor_scaling_events</span> function implementation
</div>

<div class="alert alert-block alert-info">
<b>NOTE: ⚠️</b>The <b>AlarmHigh</b> Alarm triggers scale out actions only after the threshold of <b>ConcurrentRequestsPerModel >5 </b> for 3 datapoints within <b>30 seconds</b> is breached.
</div>

In [12]:
# Trigger LLMPerf script to generate traffic to endpoint
num_concurrent_requests = 100
# LLMperf requires session credentials be passed in via environment variables.
# We'll use the current session to get these credentials.
creds = boto_session.get_credentials()
process = trigger_auto_scaling(creds, region, endpoint_name, num_concurrent_requests)
print(f"[b green]Process ID for LLMPerf: {process.pid}")

# get AlarmHigh alarm name
scaleout_alarm_name = [alarm["AlarmName"] for alarm in alarms if "AlarmHigh" in alarm["AlarmName"]][
    0
]

# Start monitoring scaling events
SLEEP_TIME = 5  # time to sleep
scaling_times = monitor_scaling_events(
    endpoint_name, scaleout_alarm_name, SLEEP_TIME, cloudwatch_client, sagemaker_client
)

# Print scaling times
console = Console()
table = print_scaling_times(scaling_times)
console.print(table)

Calling LLMPerf shell script: /Users/lucasjia/pysdk/pysdk-workspace/amazon-sagemaker-examples-lucas/     
deploy_and_monitor/sm-app_autoscaling_realtime_endpoints/trigger_autoscaling.sh

Launching LLMPerf with 100 concurrent requests

Process ID for LLMPerf: 65211

Initial instance count: 1

Tracking Alarm: 
TargetTracking-endpoint/Llama3-8B-fas-3825e-16Jul2026/variant/AllTraffic-AlarmHigh-9a27e1d2-f164-496c-8729-4bc67586
c394

Output()

### Monitor if the background process (llmperf) is completed.

In [13]:
monitor_process(process)

Process 65211 finished with return code 0

Process output:
Installing llmperf...
Created results directory.
Starting benchmarking scripts on endpoint Llama3-8B-fas-3825e-16Jul2026 ...
[33m(raylet)[0m WARNING: 48 PYTHON worker processes have been started on node: 
7b733e509a775358225e57696d6d357db34caaeb422d3aac2a2d01be with address: 127.0.0.1, which is 4x the maximum expected 
startup concurrency (12). This could be a result of using a large number of actors, tasks blocked in ray.get() 
calls, or tasks with fractional CPU requests (e.g., num_cpus=0.1) allowing high concurrency. See 
https://github.com/ray-project/ray/issues/3644 for some discussion of workarounds.
[33m(raylet)[0m WARNING: 60 PYTHON worker processes have been started on node: 
7b733e509a775358225e57696d6d357db34caaeb422d3aac2a2d01be with address: 127.0.0.1, which is 5x the maximum expected 
startup concurrency (12). This could be a result of using a large number of actors, tasks blocked in ray.get() 
calls, or tasks with fractional CPU requests (e.g., num_cpus=0.1) allowing high concurrency. See 
https://github.com/ray-project/ray/issues/3644 for some discussion of workarounds.
[33m(raylet)[0m WARNING: 72 PYTHON worker processes have been started on node: 
7b733e509a775358225e57696d6d357db34caaeb422d3aac2a2d01be with address: 127.0.0.1, which is 6x the maximum expected 
startup concurrency (12). This could be a result of using a large number of actors, tasks blocked in ray.get() 
calls, or tasks with fractional CPU requests (e.g., num_cpus=0.1) allowing high concurrency. See 
https://github.com/ray-project/ray/issues/3644 for some discussion of workarounds.
[33m(raylet)[0m WARNING: 84 PYTHON worker processes have been started on node: 
7b733e509a775358225e57696d6d357db34caaeb422d3aac2a2d01be with address: 127.0.0.1, which is 7x the maximum expected 
startup concurrency (12). This could be a result of using a large number of actors, tasks blocked in ray.get() 
calls, or tasks with fractional CPU requests (e.g., num_cpus=0.1) allowing high concurrency. See 
https://github.com/ray-project/ray/issues/3644 for some discussion of workarounds.
[33m(raylet)[0m WARNING: 96 PYTHON worker processes have been started on node: 
7b733e509a775358225e57696d6d357db34caaeb422d3aac2a2d01be with address: 127.0.0.1, which is 8x the maximum expected 
startup concurrency (12). This could be a result of using a large number of actors, tasks blocked in ray.get() 
calls, or tasks with fractional CPU requests (e.g., num_cpus=0.1) allowing high concurrency. See 
https://github.com/ray-project/ray/issues/3644 for some discussion of workarounds.
Test timed out before all requests could be completed.
\Results for token benchmark for Llama3-8B-fas-3825e-16Jul2026 queried with the sagemaker api.

inter_token_latency_s
    p25 = 0.022824333188524178
    p50 = 0.03009534305086246
    p75 = 0.037668611055808286
    p90 = 0.043620661430666785
    p95 = 0.048195586533912316
    p99 = 0.05812048938824099
    mean = 0.030138011071716812
    min = 0.003507069901495176
    max = 0.08401896235728226
    stddev = 0.011022445520421913
ttft_s
    p25 = 2.121133551998355
    p50 = 4.198648146004416
    p75 = 7.384556843502651
    p90 = 9.974934796104208
    p95 = 11.113522845555408
    p99 = 12.97838751632051
    mean = 4.892460902131542
    min = 0.23712274999707006
    max = 13.977617084005033
    stddev = 3.347891692437654
end_to_end_latency_s
    p25 = 14.78413770800762
    p50 = 20.14684487500199
    p75 = 25.797416541241546
    p90 = 30.146424586900682
    p95 = 32.92708132920961
    p99 = 38.076191065224265
    mean = 20.30472253044679
    min = 2.6341134579997743
    max = 43.931693500009715
    stddev = 7.786086363373196
request_output_throughput_token_per_s
    p25 = 26.547137634239245
    p50 = 33.227534035042595
    p75 = 43.81264304112564
    p90 = 61.36377993834832
    p95 = 79.97124675164399
    p99 = 146.96572279008416
    mean = 40.030704497655385
    min = 11.90165494495747
    max = 285.10541097583365
    stddev = 24.356

Process errors:
Cloning into 'llmperf'...
[2mUsing Python 3.12.13 environment at: /Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv[0m
[2mResolved [1m110 packages[0m [2min 391ms[0m[0m
   [36m[1mBuilding[0m[39m llmperf[2m @ 
file:///Users/lucasjia/pysdk/pysdk-workspace/amazon-sagemaker-examples-lucas/%20%20%20%20%20deploy_and_monitor/sm-a
pp_autoscaling_realtime_endpoints/llmperf[0m
      [32m[1mBuilt[0m[39m llmperf[2m @ 
file:///Users/lucasjia/pysdk/pysdk-workspace/amazon-sagemaker-examples-lucas/%20%20%20%20%20deploy_and_monitor/sm-a
pp_autoscaling_realtime_endpoints/llmperf[0m
[2mPrepared [1m1 package[0m [2min 1.26s[0m[0m
[2mUninstalled [1m1 package[0m [2min 3ms[0m[0m
[2mInstalled [1m1 package[0m [2min 59ms[0m[0m
 [31m-[39m [1mllmperf[0m[2m==0.1.0 (from 
file:///Users/lucasjia/pysdk/pysdk-workspace/amazon-sagemaker-examples-lucas/%20%20%20%20%20deploy_and_monitor/sm-a
pp_autoscaling_realtime_endpoints_step_scaling/llmperf)[0m
 [32m+[39m [1mllmperf[0m[2m==0.1.0 (from 
file:///Users/lucasjia/pysdk/pysdk-workspace/amazon-sagemaker-examples-lucas/%20%20%20%20%20deploy_and_monitor/sm-a
pp_autoscaling_realtime_endpoints/llmperf)[0m
/Users/lucasjia/pysdk/pysdk-workspace/amazon-sagemaker-examples-lucas/     
deploy_and_monitor/sm-app_autoscaling_realtime_endpoints/llmperf/token_benchmark_ray.py:139: SyntaxWarning: invalid
escape sequence '\R'
  print(f"\Results for token benchmark for {model} queried with the {llm_api} api.\n")
2026-07-16 13:44:56,624 INFO worker.py:2024 -- Started a local Ray instance.
Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits
and faster downloads.
  0%|          | 0/1000 [00:00<?, ?it/s]  0%|          | 0/1000 [00:00<?, ?it/s]  0%|          | 0/1000 [00:00<?, 
?it/s]  0%|          | 0/1000 [00:01<?, ?it/s]  0%|          | 0/1000 [00:01<?, ?it/s]  0%|          | 0/1000 
[00:02<?, ?it/s]  0%|          | 0/1000 [00:02<?, ?it/s]  0%|          | 0/1000 [00:03<?, ?it/s]  0%|          | 
0/1000 [00:03<?, ?it/s]  0%|          | 0/1000 [00:04<?, ?it/s]  0%|          | 0/1000 [00:04<?, ?it/s]  0%|       
| 0/1000 [00:05<?, ?it/s]  0%|          | 0/1000 [00:06<?, ?it/s]  0%|          | 0/1000 [00:06<?, ?it/s]  0%|     
| 0/1000 [00:06<?, ?it/s]  0%|          | 0/1000 [00:07<?, ?it/s]  0%|          | 0/1000 [00:07<?, ?it/s]  0%|     
| 0/1000 [00:08<?, ?it/s]  0%|          | 0/1000 [00:08<?, ?it/s]  0%|          | 0/1000 [00:09<?, ?it/s]  0%|     
| 0/1000 [00:09<?, ?it/s]  0%|          | 0/1000 [00:10<?, ?it/s]  0%|          | 0/1000 [00:10<?, ?it/s]  0%|     
| 0/1000 [00:11<?, ?it/s]  0%|          | 0/1000 [00:11<?, ?it/s]  0%|          | 0/1000 [00:12<?, ?it/s]  0%|     
| 0/1000 [00:12<?, ?it/s]  0%|          | 0/1000 [00:13<?, ?it/s]  0%|          | 0/1000 [00:14<?, ?it/s]  0%|     
| 0/1000 [00:14<?, ?it/s]  0%|          | 0/1000 [00:15<?, ?it/s]  0%|          | 0/1000 [00:15<?, ?it/s]  0%|     
| 0/1000 [00:16<?, ?it/s]  0%|          | 0/1000 [00:16<?, ?it/s]  0%|          | 0/1000 [00:17<?, 
?it/s][36m(SageMakerClient pid=65501)[0m Warning: You are sending unauthenticated requests to the HF Hub. Please 
set a HF_TOKEN to enable higher rate limits and faster downloads.
  0%|          | 0/1000 [00:17<?, ?it/s]  0%|          | 0/1000 [00:18<?, ?it/s]  0%|          | 0/1000 [00:18<?, 
?it/s]  0%|          | 0/1000 [00:19<?, ?it/s]  0%|          | 0/1000 [00:20<?, ?it/s]  0%|          | 0/1000 
[00:20<?, ?it/s]  0%|          | 0/1000 [00:21<?, ?it/s]  0%|          | 0/1000 [00:21<?, ?it/s]  0%|          | 
0/1000 [00:22<?, ?it/s]  0%|          | 0/1000 [00:22<?, ?it/s]  0%|          | 0/1000 [00:23<?, 
?it/s][36m(SageMakerClient pid=65618)[0m Warning: You are sending unauthenticated requests to the HF Hub. Please 
set a HF_TOKEN to enable higher rate limits and faster downloads.[32m  (Ray deduplicates logs by default. Set 
RAY_DEDUP_LOGS=0 to disable log deduplication, or see 
https://docs.ray.

## Print LLMPerf results

LLMPerf writes the results to **"results/"** directory.  `summary.json` file has the endpoint benchmarking data.

In [14]:
print_llmperf_results(num_concurrent_requests)

            LLMPerf Endpoint Metrics            
                           ╷                    
                    Metric │ Units              
 ══════════════════════════╪═══════════════════ 
       Concurrent requests │ 100                
   Avg. Input token length │ 550                
  Avg. Output token length │ 150                
  Avg. First-Time-To-Token │ 4892.46ms          
           Avg. Thorughput │ 846.40 tokens/sec  
              Avg. Latency │ 30.14ms/token      
                           ╵

### Monitor Scale-in Alarm Trigger times and scaling event times

<div class="alert alert-block alert-warning">
<b>NOTE: ⚠️</b>The <b>AlarmLow</b> Alarm triggers scale-in actions only after the threshold of <b>ConcurrentRequestsPerModel < 4.5</b> for 90 datapoints within <b>15 minutes</b> is breached.
<br/>Running the below cell with take approximately 15 minutes to complete.<br/>
</div>

In [15]:
# get AlarmHigh alarm name
scalein_alarm_name = [alarm["AlarmName"] for alarm in alarms if "AlarmLow" in alarm["AlarmName"]][0]

# Start monitoring scaling events
SLEEP_TIME = 5  # time to sleep
scaling_times = monitor_scaling_events(
    endpoint_name, scalein_alarm_name, SLEEP_TIME, cloudwatch_client, sagemaker_client
)

# Print scaling times
console = Console()
table = print_scaling_times(scaling_times)
console.print(table)

Initial instance count: 2

Tracking Alarm: 
TargetTracking-endpoint/Llama3-8B-fas-3825e-16Jul2026/variant/AllTraffic-AlarmLow-24d3971c-c89a-4b91-b395-1a2ec420f
c76

Output()

                  Scaling Times                   
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Target Instance Count ┃ Scaling Time (seconds) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│                     1 │                  81.51 │
└───────────────────────┴────────────────────────┘

## Cleanup

- Deregister scalable target. This automatically deletes associated cloudwatch alarms.
- Delete model
- Delete endpoint

In [16]:
# Deregister the scalable target for AAS
try:
    autoscaling_client.deregister_scalable_target(
        ServiceNamespace="sagemaker",
        ResourceId=resource_id,
        ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    )
    print(f"Scalable target for [b]{resource_id}[/b] deregistered. \u2705")
except autoscaling_client.exceptions.ObjectNotFoundException:
    print(f"Scalable target for [b]{resource_id}[/b] not found!.")

print("---" * 10)
# V3: predictor.delete_model()/delete_endpoint() -> delete the core Endpoint, its
# EndpointConfig, and the Model explicitly.
try:
    print(f"Deleting endpoint: [b magenta]{endpoint_name} \u2705")
    endpoint.delete()
except Exception as e:
    print(f"{e}")

try:
    print(f"Deleting endpoint config: [b magenta]{endpoint_config_name} \u2705")
    EndpointConfig.get(endpoint_config_name).delete()
except Exception as e:
    print(f"{e}")

try:
    print(f"Deleting model: [b green]{model_name} \u2705")
    Model.get(model_name).delete()
except Exception as e:
    print(f"{e}")

print("---" * 10)
print(f"Done")

Scalable target for endpoint/Llama3-8B-fas-3825e-16Jul2026/variant/AllTraffic deregistered. ✅

------------------------------

Deleting endpoint: Llama3-8B-fas-3825e-16Jul2026 ✅

[07/16/26 14:12:11] INFO     Deleting Endpoint - Llama3-8B-fas-3825e-16Jul2026                   ]8;id=1951277;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1951278;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10428\10428]8;;\

Deleting endpoint config: Llama3-8B-fas-3825e-16Jul2026 ✅

                    INFO     Deleting EndpointConfig - Llama3-8B-fas-3825e-16Jul2026             ]8;id=1951284;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1951285;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#11220\11220]8;;\

Deleting model: Llama3-8B-fas-3825e-16Jul2026 ✅

                    INFO     Deleting Model - Llama3-8B-fas-3825e-16Jul2026                      ]8;id=1951291;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1951292;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#20740\20740]8;;\

------------------------------

Done

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.


![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints|sm-app_autoscaling_realtime_endpoints.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints|sm-app_autoscaling_realtime_endpoints.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints|sm-app_autoscaling_realtime_endpoints.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints|sm-app_autoscaling_realtime_endpoints.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints|sm-app_autoscaling_realtime_endpoints.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints|sm-app_autoscaling_realtime_endpoints.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints|sm-app_autoscaling_realtime_endpoints.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints|sm-app_autoscaling_realtime_endpoints.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints|sm-app_autoscaling_realtime_endpoints.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints|sm-app_autoscaling_realtime_endpoints.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints|sm-app_autoscaling_realtime_endpoints.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints|sm-app_autoscaling_realtime_endpoints.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints|sm-app_autoscaling_realtime_endpoints.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints|sm-app_autoscaling_realtime_endpoints.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints|sm-app_autoscaling_realtime_endpoints.ipynb)